# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saadtalat111/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane (from Week 1): Refresh / Content Opportunity Scoring.**

**Task type: Ranking / scoring, built on top of a classification model.**

The question this lane answers is "which ones first?" — exactly the pattern the framing guide maps to ranking/scoring, not plain classification. Concretely:

- Underneath, I train a **binary classifier** that estimates a probability per page (e.g. "probability this page is declining").
- But the reviewer never consumes raw yes/no labels — they consume an **ordered queue**. So the classifier's output probability gets combined with a transparent baseline score into one `final_refresh_score`, and pages get **ranked** by that score.
- It is not clustering (I'm not looking for groups of similar pages — I already know what I'm trying to find: pages worth reviewing) and it is not pure classification (a bare yes/no per page ignores that reviewers only have capacity for the top N, which is a ranking problem, not a labeling problem).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Starter proxy (what I'll use first, to prove the workflow):**

```
is_declining_label = trend_direction == "down"
```

This comes from `trend_direction`, which is itself computed from `trend_pct` — a bucket calculated from the *current* 90-day window, not a future outcome. That makes it a **defined-rule proxy**, not an observed future outcome. It's fine as a first pass (it lets me build and test the whole pipeline this week), but per the lane guide it's explicitly the *weaker* option, and `trend_direction`/`trend_pct` can never be used as model features, since the label is derived from them — that would be circular.

**Stronger target (what I'll move to for the real capstone):**

```
features from a prior window (e.g. trailing 90 days)  ->  decline or recovery over a future window (e.g. next 30 days)
```

This is an **observed future outcome**, built from the warehouse's `fact_content_daily_performance` daily table, with a strict cutoff between the feature window and the target window (no metric from the target window allowed as a feature — that's the leakage check from the lane guide's validation rules).

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/saadtalat111/flyrank"
REPO_DIR = "flyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank
Starter data found. You're ready.


In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The starter proxy target, sketched directly (never used as a feature -- label only)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Rows: {len(df):,}")
print(f"Positive rate (is_declining_label == 1): {df['is_declining_label'].mean():.1%}")
print()
print(df[["content_id", "client_id", "impressions_90d", "avg_position",
          "ctr", "trend_direction", "is_declining_label"]].head(5).to_string(index=False))


Rows: 30,000
Positive rate (is_declining_label == 1): 54.2%

          content_id         client_id  impressions_90d  avg_position  ctr trend_direction  is_declining_label
content_304f48230142 client_f369cb89fc             3803          10.6 0.76            down                   1
content_a1fb4e703a9e client_4e07408562            15320          20.3 0.05            down                   1
content_9aa793d4d895 client_7f2253d7e2            12581          36.5 0.09            down                   1
content_331d6c4de07b client_19581e27de            11751           6.2 0.49          stable                   0
content_d99b7a2d90ca client_3fdba35f04            19140          44.0 0.13            down                   1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: precision@K, specifically precision@50** (K set by realistic reviewer capacity — the lane guide suggests picking K to match "the real decision capacity").

**Why this metric and not accuracy:** a reviewer works down a ranked list and only ever gets through the first K pages. Accuracy across all 30,000 rows rewards being right about pages nobody will ever look at. Precision@50 asks the only question that matters operationally: *of the top 50 pages the system says to review first, how many actually turned out positive?*

**What "good" means here, in a real number:** the starter pipeline already gives me a baseline to beat. A fixed rule (`baseline_rules`) scores precision@50 = 0.240 — about 12 of the top 50 are right. That's my floor. Anything I build needs to clear that, on a client-holdout split (whole clients withheld from training), or it hasn't earned its complexity.

I'll also track **ROC AUC** and **average precision** as secondary metrics (useful for comparing models during development), but precision@K is the one I'll defend if asked "did this work?" — because it's the one that matches how the output actually gets used.

In [4]:
with open("outputs/model_report.md") as f:
    report = f.read()

start = report.index("## Model Comparison")
end = report.index("## Final Queue")
print(report[start:end].strip())
print()
print("Floor to beat: baseline_rules precision@50 = 0.240 (about 12 of the top 50 right).")


## Model Comparison

Best model: `random_forest` selected by `precision_at_50`.

| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |

Floor to beat: baseline_rules precision@50 = 0.240 (about 12 of the top 50 right).


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one pseudonymized content item (`content_id`)**, described by its trailing-90-day search and engagement metrics, plus metadata like word count, content type, and intent. It is NOT one row per client, and it is NOT one row per day — each page appears exactly once in the starter slice (deduplicated by `content_id`, per the lane guide).

In [5]:
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique content_id values: {df['content_id'].nunique():,}  (should equal row count -> one row per page)")
print(f"Unique client_id values:  {df['client_id'].nunique()}")
print()
print("Sample row (one page, all fields):")
df.sample(1, random_state=7).T


Shape: 30,000 rows x 45 columns
Unique content_id values: 30,000  (should equal row count -> one row per page)
Unique client_id values:  32

Sample row (one page, all fields):


,1252
content_id,content_a8c35eeef547
client_id,client_19581e27de
search_volume,70.0
competition,0.05
competition_level,LOW
cpc,0.0
content_type,keyword article
main_intent,transactional
word_count,NaN
char_count,NaN


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The starter's own hand-written rule (`baseline_refresh_score`) is a weighted sum of four sub-scores — visibility, freshness risk, position opportunity, depth gap — with fixed weights (0.40 / 0.30 / 0.25 / 0.05) chosen once, up front, for every page. But a page's real "worth reviewing" signal comes from many overlapping factors that don't combine the same way for every page: a stale page with huge demand needs different weighting than a fresh page with a sudden CTR drop, and that mix likely shifts by content type, intent, and client. Writing a bigger and bigger pile of if-statements to capture every combination doesn't scale and isn't testable — a model can instead learn how much each signal actually matters, directly from evidence, and that weighting can be validated instead of just asserted.

This isn't a hypothesis — the starter pipeline already tested it on this exact lane and label:

In [6]:
print(report[start:end].strip())
print()
print("The random forest recovers about 3x more true positives at the same review budget")
print("(37 of the top 50, vs 12 of the top 50 for the fixed rule) -- using client-holdout")
print("validation, so this isn't the model just memorizing clients it already saw.")


## Model Comparison

Best model: `random_forest` selected by `precision_at_50`.

| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |

The random forest recovers about 3x more true positives at the same review budget
(37 of the top 50, vs 12 of the top 50 for the fixed rule) -- using client-holdout
validation, so this isn't the model just memorizing clients it already saw.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.